In [1]:
# Create the finalized train/test CSV files directly from the original raw datasets.

import pandas as pd

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

train["product_category_clean"] = train["product_category"].str.strip().str.lower()
test["product_category_clean"] = test["product_category"].str.strip().str.lower()

final_features = [
    "product_code",
    "product_weight_kg",
    "fat_content",
    "shelf_visibility",
    "product_price",
    "store_code",
    "store_age_years",
    "store_size",
    "store_location_tier",
    "store_format",
    "product_category_clean",
]

cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

train_final = train[final_features + ["total_sales"]].copy()
test_final = test[final_features].copy()

for col in cat_features:
    train_final[col] = train_final[col].fillna("__MISSING__").astype(str)
    test_final[col] = test_final[col].fillna("__MISSING__").astype(str)

train_final.to_csv("../data/train_final.csv", index=False)
test_final.to_csv("../data/test_final.csv", index=False)

print("train_final:", train_final.shape)
print("test_final:", test_final.shape)

train_final: (6818, 12)
test_final: (1705, 11)


In [2]:
# Train the frozen champion Ordered CatBoost model directly from the finalized training CSV.

import pandas as pd
from catboost import CatBoostRegressor

# Load the finalized training dataset
train_final = pd.read_csv("../data/train_final.csv")

# Separate predictors and target
X_train_final = train_final.drop(columns=["total_sales"])
y_train_final = train_final["total_sales"]

# Locked categorical features
cat_features = [
    "product_code",
    "fat_content",
    "product_category_clean",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

# Frozen champion configuration
final_model = CatBoostRegressor(
    iterations=300,
    learning_rate=0.025,
    depth=4,
    l2_leaf_reg=1,
    boosting_type="Ordered",
    random_strength=1.0,
    loss_function="RMSE",
    random_seed=42,
    verbose=False,
    thread_count=-1,
)

# Train on all available training observations
final_model.fit(
    X_train_final,
    y_train_final,
    cat_features=cat_features,
)

print("Model trained successfully.")
print(f"Training rows: {len(X_train_final)}")
print(f"Features: {X_train_final.shape[1]}")
print(f"Iterations: {final_model.get_params()['iterations']}")
print(f"Learning rate: {final_model.get_params()['learning_rate']}")
print(f"Depth: {final_model.get_params()['depth']}")
print(f"L2 leaf regularization: {final_model.get_params()['l2_leaf_reg']}")
print(f"Boosting type: {final_model.get_params()['boosting_type']}")

Model trained successfully.
Training rows: 6818
Features: 11
Iterations: 300
Learning rate: 0.025
Depth: 4
L2 leaf regularization: 1
Boosting type: Ordered


In [3]:
# Generate predictions for every test observation using the fully trained champion model.

import pandas as pd
import numpy as np

# Load the finalized test dataset
test_final = pd.read_csv("../data/test_final.csv")

# Generate predictions
test_predictions = final_model.predict(test_final)

# Basic prediction checks
print("Prediction count:", len(test_predictions))
print("Missing predictions:", np.isnan(test_predictions).sum())
print("Minimum prediction:", test_predictions.min())
print("Maximum prediction:", test_predictions.max())
print("Mean prediction:", test_predictions.mean())
print("Median prediction:", np.median(test_predictions))

Prediction count: 1705
Missing predictions: 0
Minimum prediction: 7.296992603987746
Maximum prediction: 5938.934543869664
Mean prediction: 2210.526817627066
Median prediction: 2139.8934926324473


In [4]:
# Build the competition submission file with the original test IDs and final model predictions.

import pandas as pd

# Load original test data to preserve the official test IDs
test_original = pd.read_csv("../data/test.csv")

submission = pd.DataFrame({
    "id": test_original["id"],
    "total_sales": test_predictions,
})

submission.to_csv("../data/submission.csv", index=False)

print("Shape:", submission.shape)
print("\nColumns:", submission.columns.tolist())
print(submission.head())
print("\nMissing values:")
print(submission.isna().sum())

Shape: (1705, 2)

Columns: ['id', 'total_sales']
          id  total_sales
0  row_00009  2842.297224
1  row_00015  4015.948994
2  row_00019  3067.439601
3  row_00020  3324.373837
4  row_00023  2490.996528

Missing values:
id             0
total_sales    0
dtype: int64


In [5]:
# Verify that the submission IDs exactly match the original test IDs and that the submission is structurally valid.

import pandas as pd

test_original = pd.read_csv("../data/test.csv")
submission = pd.read_csv("../data/submission.csv")

ids_match = submission["id"].equals(test_original["id"])
ids_unique = submission["id"].is_unique
row_count_match = len(submission) == len(test_original)
columns_correct = submission.columns.tolist() == ["id", "total_sales"]
no_missing = submission[["id", "total_sales"]].isna().sum().sum() == 0

print("Row count matches:", row_count_match)
print("IDs match original test set:", ids_match)
print("IDs are unique:", ids_unique)
print("Columns are correct:", columns_correct)
print("No missing values:", no_missing)

print("\nSubmission ready:", all([
    row_count_match,
    ids_match,
    ids_unique,
    columns_correct,
    no_missing,
]))

Row count matches: True
IDs match original test set: True
IDs are unique: True
Columns are correct: True
No missing values: True

Submission ready: True
